<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/00b_painel_gerencial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: apêndice do capítulo 0, painel gerencial

Continuação dos capítulos 1 a 5, mas os dados aqui alimentam o **capítulo 0** (a vitrine/convite), não um capítulo novo. Três exportações: frete em % do valor por estado e cidade (com faturamento e frete pago juntos, pro mapa com métrica à escolha e drill-down), faturamento mensal (pro painel com filtro de período), e os coeficientes de uma regressão linear pro simulador "e se" de frete. Não precisa de T4, só pandas e um `LinearRegression` no fim.

In [2]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

pedido = order_items.groupby('order_id').agg(
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
).reset_index()
pedido = pedido.merge(orders[['order_id', 'customer_id', 'order_purchase_timestamp']], on='order_id', how='left')
pedido = pedido.merge(customers[['customer_id', 'customer_state', 'customer_city']], on='customer_id', how='left')
pedido['freight_pct'] = pedido['freight_value'] / pedido['price']
print(f"pedidos agregados: {pedido.shape}")

pedidos agregados: (98666, 8)


## Frete %, faturamento e frete pago, por estado e cidade

In [3]:
por_estado = pedido.groupby('customer_state').agg(
    freight_pct=('freight_pct', 'mean'),
    revenue=('price', 'sum'),
    freight_paid=('freight_value', 'sum'),
    orders=('order_id', 'nunique'),
).reset_index()

resultado = {}
for _, row in por_estado.iterrows():
    estado = row['customer_state']
    cidades_do_estado = pedido[pedido['customer_state'] == estado]
    top_cidades = (
        cidades_do_estado.groupby('customer_city').agg(
            freight_pct=('freight_pct', 'mean'),
            revenue=('price', 'sum'),
            freight_paid=('freight_value', 'sum'),
            orders=('order_id', 'nunique'),
        )
        .sort_values('orders', ascending=False)
        .head(10)
        .reset_index()
    )
    resultado[estado] = {
        'freight_pct': round(float(row['freight_pct']), 4),
        'revenue': round(float(row['revenue']), 2),
        'freight_paid': round(float(row['freight_paid']), 2),
        'orders': int(row['orders']),
        'cities': [
            {
                'city': c['customer_city'].title(),
                'freight_pct': round(float(c['freight_pct']), 4),
                'revenue': round(float(c['revenue']), 2),
                'freight_paid': round(float(c['freight_paid']), 2),
                'orders': int(c['orders']),
            }
            for _, c in top_cidades.iterrows()
        ],
    }

with open('state-city-metrics.json', 'w', encoding='utf-8') as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"{len(resultado)} estados exportados")
print(por_estado.sort_values('revenue', ascending=False).head(5))

27 estados exportados
   customer_state  freight_pct     revenue  freight_paid  orders
25             SP     0.251448  5202955.05     718723.07   41375
18             RJ     0.314826  1824092.67     305589.31   12762
10             MG     0.315411  1585308.03     270853.46   11544
22             RS     0.340145   750304.02     135522.74    5432
17             PR     0.334186   683083.76     117851.68    4998


## Faturamento por mês

In [4]:
pedido['mes'] = pedido['order_purchase_timestamp'].dt.to_period('M').astype(str)
por_mes = pedido.groupby('mes').agg(
    revenue=('price', 'sum'),
    freight_paid=('freight_value', 'sum'),
    orders=('order_id', 'nunique'),
).reset_index()

serie_mensal = [
    {
        'month': row['mes'],
        'revenue': round(float(row['revenue']), 2),
        'freight_paid': round(float(row['freight_paid']), 2),
        'orders': int(row['orders']),
    }
    for _, row in por_mes.iterrows()
]
with open('revenue-by-month.json', 'w', encoding='utf-8') as f:
    json.dump(serie_mensal, f, ensure_ascii=False, indent=2)
print(f"{len(serie_mensal)} meses exportados")

24 meses exportados


## Regressão linear pro frete, o motor do simulador "e se"

O capítulo 4 já treinou uma Regressão Linear pro cenário de frete (R² 0,53), com as mesmas features físicas do produto mais a distância. Aqui eu refaço o treino só pra extrair os coeficientes num formato pequeno o suficiente pra rodar no navegador: `frete_previsto = intercepto + Σ coeficiente_i × valor_i`. Exporto também a faixa real (percentil 5 a 95, pra não deixar o slider ir a valores absurdos fora do que o dataset realmente tem) de cada feature.

In [5]:
geo_media = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

sellers = pd.read_csv('olist_sellers_dataset.csv')

m = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
)
m['customer_zip_code_prefix'] = m['customer_zip_code_prefix'].astype('int64')
m['seller_zip_code_prefix'] = m['seller_zip_code_prefix'].astype('int64')
geo_media['geolocation_zip_code_prefix'] = geo_media['geolocation_zip_code_prefix'].astype('int64')
m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}), on='customer_zip_code_prefix', how='left')
m = m.merge(geo_media.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix', 'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}), on='seller_zip_code_prefix', how='left')
m['distance_km'] = haversine(m['cust_lat'], m['cust_lng'], m['sell_lat'], m['sell_lng'])

s3 = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'freight_value', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'distance_km']]
    .dropna()
)
features_3 = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'distance_km']
X3, y3 = s3[features_3], s3['freight_value']
X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.2, random_state=42)

modelo_frete = LinearRegression()
modelo_frete.fit(X3tr, y3tr)
r2 = r2_score(y3te, modelo_frete.predict(X3te))

coeficientes = {
    'intercept': round(float(modelo_frete.intercept_), 4),
    'r2': round(float(r2), 4),
    'coefficients': {f: round(float(c), 6) for f, c in zip(features_3, modelo_frete.coef_)},
    'ranges': {
        f: [round(float(s3[f].quantile(0.05)), 2), round(float(s3[f].quantile(0.95)), 2)]
        for f in features_3
    },
}
with open('freight-linear-coefficients.json', 'w', encoding='utf-8') as f:
    json.dump(coeficientes, f, ensure_ascii=False, indent=2)
print(json.dumps(coeficientes, indent=2))

{
  "intercept": 6.1466,
  "r2": 0.5322,
  "coefficients": {
    "product_weight_g": 0.002412,
    "product_length_cm": 0.036606,
    "product_height_cm": 0.064423,
    "product_width_cm": 0.01912,
    "distance_km": 0.010627
  },
  "ranges": {
    "product_weight_g": [
      125.0,
      9750.0
    ],
    "product_length_cm": [
      16.0,
      61.0
    ],
    "product_height_cm": [
      3.0,
      44.0
    ],
    "product_width_cm": [
      11.0,
      45.0
    ],
    "distance_km": [
      16.44,
      2098.41
    ]
  }
}
